# EX03. [추가연습] 불균형 데이터 실전 문제 - 고객 이탈 예측

> 클래스 불균형이 심한 고객 이탈(Churn) 데이터로, accuracy만으로는 부족한 이유와 대응법(`06_머신러닝_분류`/`08_딥러닝`에서 배운 class_weight, SMOTE)을 실제 불균형 데이터에 제대로 적용해봅니다.

### 사용 데이터
`data/telco_churn.csv` - 통신사 고객 7,043명의 서비스 이용 정보로 이탈(Churn) 여부(Yes/No, 약 26.5% : 73.5%)를 예측

## 목차
1. 데이터 로딩 & EDA
2. 전처리
3. 머신러닝 모델링 & 평가지표 심화
4. 딥러닝 모델링
5. 종합 실전 문제

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/telco_churn.csv')
df.head()

## 1. 데이터 로딩 & EDA

### 📖 이론 설명
이탈 비율을 가장 먼저 확인합니다 - 이탈(Yes)이 소수 클래스입니다. `TotalCharges`는 숫자여야 하는데 `info()`를 보면 `object`(문자열) 타입인 것도 미리 확인해둡니다(2절에서 원인을 찾아 고칩니다).

### 💻 예제 코드

In [ ]:
df.info()

print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True))

sns.countplot(x='Churn', data=df)
plt.title('Churn distribution (imbalanced)')
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `Contract` 컬럼별 이탈률(Churn=='Yes' 비율)을 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean())  # 그룹별로 'Yes' 비율을 직접 계산(Churn이 아직 0/1로 인코딩되기 전이라 mean() 대신 조건식을 사용)
```

</details>

## 2. 전처리

### 📖 이론 설명
`TotalCharges`가 문자열인 이유는 값이 없는 자리에 **빈 문자열(' ')**이 들어있기 때문입니다(NaN이 아니라서 `isnull()`로는 안 보입니다!). `replace()`로 공백을 NaN으로 바꾼 뒤 `astype(float)`로 변환합니다. 또한 `customerID`는 예측에 의미가 없는 식별자이므로 제거하고, 나머지 범주형(Yes/No, 문자열) 컬럼은 원-핫 인코딩합니다.

### 💻 예제 코드

In [ ]:
# 1) TotalCharges 정리
print((df['TotalCharges'] == ' ').sum(), '개의 공백 값')
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan).astype(float)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 2) 불필요 컬럼 제거
df = df.drop(columns=['customerID'])

# 3) 타겟 레이블 숫자화
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 4) 나머지 범주형 컬럼 원-핫 인코딩
cat_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df.info()

### ✍️ TODO: 실전 문제

**문제 1.** 전처리 완료 후 `df`에 결측치가 남아있지 않은지 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(df.isnull().sum().sum())  # sum()을 두 번 써서 컬럼별 결측치 합계를 다시 전체 합계로(0이면 결측치 없음)
```

</details>

## 3. 머신러닝 모델링 & 평가지표 심화

### 📖 이론 설명
먼저 **아무 튜닝도 없이** 모델을 학습시켜 accuracy는 높지만 recall이 낮은 함정을 직접 확인합니다. 그다음 `class_weight='balanced'`를 적용해 recall이 어떻게 바뀌는지 비교합니다.

### 💻 예제 코드

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 기본 모델: accuracy는 높아 보이지만 recall이 낮다
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_s, y_train)
pred = rf.predict(X_test_s)
print('기본 모델 - accuracy:', accuracy_score(y_test, pred), ' recall:', recall_score(y_test, pred))

# class_weight='balanced' 적용
rf_bal = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_bal.fit(X_train_s, y_train)
pred_bal = rf_bal.predict(X_test_s)
print('balanced 모델 - accuracy:', accuracy_score(y_test, pred_bal), ' recall:', recall_score(y_test, pred_bal))

cm = confusion_matrix(y_test, pred_bal)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('class_weight=balanced Confusion Matrix')
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `imblearn.SMOTE`로 `X_train_s`, `y_train`을 오버샘플링한 뒤 `RandomForestClassifier`를 다시 학습시켜 recall을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_over, y_over = smote.fit_resample(X_train_s, y_train)  # 학습 데이터에만 오버샘플링 적용(테스트 데이터는 그대로 둬야 실제 성능을 왜곡 없이 평가 가능)
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_over, y_over)
pred_smote = rf_smote.predict(X_test_s)
print(recall_score(y_test, pred_smote))
print(precision_score(y_test, pred_smote))  # recall이 오르는 대신 precision이 떨어지는 트레이드오프가 있는지 함께 확인
```

</details>

## 4. 딥러닝 모델링

### 📖 이론 설명
딥러닝에서도 동일하게 SMOTE로 학습 데이터를 오버샘플링한 뒤 학습시켜 recall 변화를 관찰합니다.

### 💻 예제 코드

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from imblearn.over_sampling import SMOTE

tf.random.set_seed(100)
smote = SMOTE(random_state=42)
X_over, y_over = smote.fit_resample(X_train_s, y_train)

model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train_s.shape[1],)))
model.add(Dropout(0.3))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_over, y_over, epochs=50, batch_size=32,
                     validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)

dl_pred = (model.predict(X_test_s).flatten() > 0.5).astype(int)
print('DL(SMOTE) recall:', recall_score(y_test, dl_pred))
print('DL(SMOTE) precision:', precision_score(y_test, dl_pred))

## 5. 종합 실전 문제

**불균형 데이터 미니 모의고사입니다.**

**문제 1.** `LogisticRegression(class_weight='balanced', max_iter=1000)`을 학습시켜 f1-score를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.linear_model import LogisticRegression
logi = LogisticRegression(class_weight='balanced', max_iter=1000)  # class_weight='balanced': 소수 클래스(이탈)의 오분류에 더 큰 패널티를 줌
logi.fit(X_train_s, y_train)
print(f1_score(y_test, logi.predict(X_test_s)))
```

</details>

**문제 2.** `class_weight` 없는 기본 모델과 `balanced` 모델의 recall을 나란히 비교하는 표를 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
compare = pd.DataFrame({
    'no_weight': [recall_score(y_test, pred), precision_score(y_test, pred)],
    'balanced': [recall_score(y_test, pred_bal), precision_score(y_test, pred_bal)],
}, index=['recall', 'precision'])  # 두 모델의 recall/precision을 표로 나란히 정리하면 트레이드오프를 한눈에 비교하기 좋음
print(compare)
```

</details>